<a href="https://colab.research.google.com/github/honeylouluzon/Guided-Project-AI-ML-DataScience/blob/main/Embankment_Calculator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title Interactive Form { display-mode: "form" }

import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import math
from datetime import datetime
from google.colab import files
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML, Markdown
import time

# Import the Python SDK
import google.generativeai as genai
# Used to securely store your API key
from google.colab import userdata

# Try to load API key
try:
    GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
except Exception as e:
    GOOGLE_API_KEY = None

#INITAL VALUE
mainThickness = widgets.IntText(value=10,description='Depths/Heights (m):',disabled=False, style={'description_width': 'initial'}, layout=widgets.Layout(width='80%'))
mainWidth = widgets.IntText(value=10,description='Width (m):',disabled=False, style={'description_width': 'initial'}, layout=widgets.Layout(width='80%'))
mainLength = widgets.IntText(value=10,description='Length (m):',disabled=False, style={'description_width': 'initial'}, layout=widgets.Layout(width='80%'))
zoneCType_Option = ["Tree Area", "Grass Area"]
zoneCType = widgets.Dropdown(options=zoneCType_Option, value="Tree Area", description='Zone C Type:', style={'description_width': 'initial'}, layout=widgets.Layout(width='80%'))
zone_C_total = widgets.FloatSlider(value=0.5, min=0.3, max=0.5, step=0.05, description="Zone C (Depths/Heights) (m):", style={'description_width': 'initial'}, layout=widgets.Layout(width='80%'))
areaType_Option = ["Soils", "Coarse Rockfill", "Underwater"]
areaType = widgets.Dropdown(options=areaType_Option, value="Soils", description='Type of Area:', style={'description_width': 'initial'}, layout=widgets.Layout(width='80%'))
                                #Soils (fine/engineered fill): 150–300 mm per loose lift before compaction (default 200 mm).
                                #Coarse rockfill: 300–600 mm per lift (default 400 mm).
                                #Underwater / very soft foundations: thicker rock blanket layers often required (e.g., 500–1,500 mm) — treat as a special case based on geotechnical input.
liftThickness = widgets.FloatSlider(value=0.5, min=0.15, max=0.3, step=0.05, description='Lift Thickness \nbefore Compaction (m):', style={'description_width': 'initial'}, layout=widgets.Layout(width='80%'))
truckCapacity = widgets.IntText(value=22,description='Truck Capacity (cubic meter):',disabled=False, style={'description_width': 'initial'}, layout=widgets.Layout(width='80%'))
actualDrop = widgets.IntText(value=0,description='Count of Actual Drops (Grayed):',disabled=False, style={'description_width': 'initial'}, layout=widgets.Layout(width='80%'))
generate = widgets.Button(description="Generate Image", style={'description_width': 'initial'})
aiswitch = widgets.ToggleButton(value=False, description='AI(Disabled)', icon='robot')

#DOWNLOADING VALUES
download = widgets.Button(description="Download Document File", style={'description_width': 'auto'})
file_name = f"embarkment_{datetime.now().strftime('%Y%m%d')}.png"
filename = widgets.Text(value=f"embarkment_{datetime.now().strftime('%Y%m%d')}.doc",description='Filename:',disabled=False, style={'description_width': 'initial'}, layout=widgets.Layout(width='80%'))
details = widgets.Textarea(value="",description='Details:',disabled=False, style={'description_width': 'initial'}, layout=widgets.Layout(width='80%', height='250px'))
recommendation = widgets.Textarea(value="",description='Recommendation:',disabled=True, style={'description_width': 'initial'}, layout=widgets.Layout(width='80%', height='250px'))


# Check if key exists
if not GOOGLE_API_KEY:
  gemini_model = None
  aiswitch.disabled = True
  aiswitch.tooltip = "No API key found — AI features disabled"
else:
    try:
      genai.configure(api_key=GOOGLE_API_KEY)
      # Initialize the Gemini API
      gemini_model = genai.GenerativeModel('gemini-2.5-flash-preview-05-20')
      aiswitch.tooltip = "Click to enable or disable AI"
    except Exception as e:
      gemini_model = None
      aiswitch.disabled = True
      aiswitch.tooltip = "No API key found — AI features disabled"


#LOAD DOCS MODULE
# 1. Install / upgrade required packages
!pip install python-docx

# Clear output after install to keep the cell clean
clear_output(wait=True)

# 1. Install / upgrade required packages
#!pip install -q --upgrade torch
#clear_output(wait=True)
#!pip install -q transformers triton==3.4 kernels
#clear_output(wait=True)
#!pip uninstall -q torchvision torchaudio -y


# Clear output after install to keep the cell clean
clear_output(wait=True)

#Load Docs
from docx import Document
from docx.shared import Inches, Pt
from docx.enum.text import WD_ALIGN_PARAGRAPH

## 2. Load the model & tokenizer
#from transformers import AutoModelForCausalLM, AutoTokenizer
#model_id = "openai/gpt-oss-20b"
#tokenizer = AutoTokenizer.from_pretrained(model_id)
#model = AutoModelForCausalLM.from_pretrained(
#    model_id,
#    torch_dtype="auto",
#    device_map="cpu"
#)

spinner_html = """
<div class="loader"></div>
<style>
.loader {
  border: 6px solid #f3f3f3;
  border-top: 6px solid #3498db;
  border-radius: 50%;
  width: 40px;
  height: 40px;
  animation: spin 1s linear infinite;
  margin: 10px auto;
}
@keyframes spin {
  0% { transform: rotate(0deg); }
  100% { transform: rotate(360deg); }
}
</style>
"""

# Title above the input area
display(widgets.HTML("<Center><h2>Embarkment Calculator</h2></Center>"))
display(widgets.HTML("<Center><a href='https://chatgpt.com/' target='_blank'>Co-Created by ChatGPT</a></Center>"))
display(widgets.HTML("<Center><a href='https://drive.google.com/file/d/1hBGz7DkKIq_eJNv2VBrFk-hlh2OVIY0E/view' target='_blank'>Click here for Tutorial or Demo</a></Center>"))
display(widgets.HTML("<Center>Try also this App: <a href='https://embankment-calculator.lovable.app/' target='_blank'>Embankment Calculator</a></Center>"))
display(widgets.HTML("<h2></h2>"))

# Show widgets
display(mainThickness, mainWidth, mainLength, zoneCType, zone_C_total, areaType, liftThickness, truckCapacity, actualDrop, aiswitch, generate)


def on_aiswitch_change(change):
  if change['new']:
    aiswitch.description = 'AI(Enabled)'
    aiswitch.button_style = 'success'
  else:
    aiswitch.description = 'AI(Disabled)'
    aiswitch.button_style = ''

def on_zoneCType_change(change):
    # Adjust range based on Zone C Type
    if change['new'].startswith("Grass Area"):
        zone_C_total.min, zone_C_total.max = 0.15, 0.2
        zone_C_total.step = 0.01
        zone_C_total.value = 0.2
    else:
        zone_C_total.max, zone_C_total.min  = 0.5, 0.3
        zone_C_total.step = 0.05
        zone_C_total.value = 0.5

def on_areaType_change(change):
    # Adjust range based on Zone C Type
    if change['new'].startswith("Soils"):
      liftThickness.default = 0.3
      liftThickness.min, liftThickness.max = 0.15, 0.3
      liftThickness.step = 0.05
    elif change['new'].startswith("Coarse Rockfill"):
      liftThickness.max, liftThickness.min  = 0.6, 0.3
      liftThickness.step = 0.05
      liftThickness.default = 0.4
    else:
      liftThickness.max, liftThickness.min = 1.5, 0.5
      liftThickness.step = 0.05
      liftThickness.default = 0.5

def show_spinner():
    display(HTML(spinner_html))

def hide_spinner():
    display(HTML("<style>.loader { display:none; }</style>"))

# Zone definitions (total thickness in meters, width in meters, color)
def zone_total(zone):
  zone_A_total = mainThickness.value * 0.25
  zone_B_total = mainThickness.value - zone_A_total - zone_C_total.value
  if zone == "A":
    return zone_A_total
  elif zone == "B":
    return zone_B_total
  else:
    return zone_C_total.value

# Zone Color
def zone_color(zone):
  if zone == "A":
    return "#C4A484"  # tan
  elif zone == "B":
    return "#f4a261"  # orange
  elif zone == "C":
    return "#e9c46a"  # yellow
  else:
    return "#D3D3D3"  # Light Gray

# Function to auto-calculate number of lifts
def lifts_counts(zoneTotal):
    return math.ceil(zoneTotal/liftThickness.value)

# Function to auto-generate lifts dynamically
def generate_lifts(zone):
    thickness = zone_total(zone) / lifts_counts(zone_total(zone))
    lifts = []
    for i in range(lifts_counts(zone_total(zone))):
        name = f"{zone}{i+1}"
        lifts.append((name, thickness))
    return lifts

###Compacted:Compaction (loose/compacted) ratios: rock = 1.30, soil = 1.15, top = 1.10.
def dumpThickness(zone):
    if zone == "A":
      return liftThickness.value * 1.30
    elif zone == "B":
      return liftThickness.value * 1.15
    else:
      return liftThickness.value * 1.10

def dumpWidth(zone):
  return (truckCapacity.value/dumpThickness(zone))** (1/3)

def dumpCountW(zone):
  return math.ceil(mainWidth.value/dumpWidth(zone))

def dumpCountL(zone):
  return math.ceil(mainLength.value/dumpWidth(zone))

def plannedDrops(zone):
  return lifts_counts(zone_total(zone)) * dumpCountW(zone) * dumpCountL(zone)

# Function to draw a stacked zone with custom width + color
def draw_zone(x, y_start, zone_label, zone_width, ax, zone, aDrop):
    tempADrop = aDrop
    y = y_start
    total_height = sum(h for _, h in generate_lifts(zone))
    # Outer box
    outer = Rectangle(((x - (zone_width/2)) - 0.4, y_start), zone_width + 0.4, total_height,   #Rectangle((x - zone_width/2, y_start)
                      fill=False, linewidth=2, edgecolor="black")
    ax.add_patch(outer)

    # Zone label
    ax.text(x - (zone_width/2) - 2.3, y_start + (total_height/2) - 0.2,
            f" {zone_label} \n Total Height: {total_height:.2f} m \n Total Width: {zone_width:.2f} m",
            ha="left", va="bottom", fontsize=8, weight='bold')

    # Draw each lift with fill color
    for name, h in generate_lifts(zone):
      start_x = x - (zone_width/2)
      tempTotalW=0
      tempDumpWidth=dumpWidth(zone)
      tempdumpCountW=dumpCountW(zone)
      tempdumpCountL=dumpCountL(zone)

      for i in range(math.ceil(tempdumpCountW)):
        tempDumpWidth_0=tempDumpWidth
        if (tempTotalW + tempDumpWidth) > mainWidth.value:
          tempDumpWidth = tempDumpWidth - (tempTotalW + tempDumpWidth - zone_width)
        #print(f"Zone {zone}: tempDumpWidth={tempDumpWidth:.2f}, tempTotalW={tempTotalW:.2f}, mainWidth={mainWidth.value:.2f}")
        if tempADrop < 1:
          rect = Rectangle((start_x, y), tempDumpWidth, h,
                          facecolor=zone_color(zone), edgecolor="black", linewidth=1)
          ax.add_patch(rect)
          ax.text(start_x +(tempDumpWidth/2), y + h/2, f"~{tempdumpCountL:.0f} drops within {tempDumpWidth:.2f} m \n in every {tempDumpWidth_0:.2f} m", ha="center", va="center", fontsize=6) # Added format specifiers for better readability
        elif tempADrop < tempdumpCountL:
          rect = Rectangle((start_x, y), tempDumpWidth, h,
                          facecolor=zone_color(""), edgecolor="black", linewidth=1)
          ax.add_patch(rect)
          ax.text(start_x +(tempDumpWidth/2), y + h/2, f"{tempADrop:.0f}/{tempdumpCountL:.0f} dropped within {tempDumpWidth:.2f} m \n in every {tempDumpWidth_0:.2f} m", ha="center", va="center", fontsize=6) # Added format specifiers for better readability
        else:
          rect = Rectangle((start_x, y), tempDumpWidth, h,
                          facecolor=zone_color(""), edgecolor="black", linewidth=1)
          ax.add_patch(rect)
          ax.text(start_x +(tempDumpWidth/2), y + h/2, f"{tempdumpCountL:.0f} dropped within {tempDumpWidth:.2f} m \n in every {tempDumpWidth_0:.2f} m", ha="center", va="center", fontsize=6) # Added format specifiers for better readability

        tempADrop = tempADrop - tempdumpCountL
        tempTotalW=tempTotalW+tempDumpWidth
        start_x = start_x + tempDumpWidth

      ax.text(mainWidth.value +(x - zone_width/2) + 0.3, y + h/2, f"{name} | {h:.3f} m height", ha="left", va="center", fontsize=7)
      y += h
    return total_height

def on_generate(b):
  clear_output(wait=True)
  display(widgets.HTML("<Center><h2>Embarkment Calculator</h2></Center>"))
  display(widgets.HTML("<Center><a href='https://chatgpt.com/' target='_blank'>Co-Created by ChatGPT</a></Center>"))
  display(widgets.HTML("<Center><a href='https://drive.google.com/file/d/1hBGz7DkKIq_eJNv2VBrFk-hlh2OVIY0E/view' target='_blank'>Click here for Tutorial or Demo</a></Center>"))
  display(widgets.HTML("<Center>Try also this App: <a href='https://embankment-calculator.lovable.app/' target='_blank'>Embankment Calculator</a></Center>"))
  display(widgets.HTML("<h2></h2>"))
  display(mainThickness, mainWidth, mainLength, zoneCType, zone_C_total, areaType, liftThickness, truckCapacity, actualDrop, aiswitch, generate)
  show_spinner()
  # Create figure
  fig, ax = plt.subplots(figsize=(mainWidth.value+2, mainThickness.value+3.2))

  # Draw zones stacked vertically
  x_center = (mainWidth.value)/2
  y_base = 0.0
  hA = draw_zone(x_center, y_base, "ZONE A (Rockfill)", mainWidth.value, ax, "A", actualDrop.value)
  hB = draw_zone(x_center, y_base + hA, "ZONE B (Soil)", mainWidth.value, ax, "B", actualDrop.value-plannedDrops("A"))
  hC = draw_zone(x_center, y_base + hA + hB, "ZONE C (Top Surface)", mainWidth.value, ax, "C", actualDrop.value-plannedDrops("A")-plannedDrops("B"))

  plannedDrops_total = plannedDrops("A") + plannedDrops("B") + plannedDrops("C")

  Detail = (f"----D A T A   S U M M A R Y----\n\nDIMENSION\nHeight/Depth: {mainThickness.value:.2f} m\nWidth: {mainWidth.value:.2f} m\nLength: {mainLength.value:.2f} m\n\n"
          f"ZONE C or TOP FILL = {zoneCType.value} \nConservative Approach: 150–200 mm for grass areas(default 200 mm)\n300–500 mm for planted/tree areas(default 500 mm)\nCustom(default 300 mm))\n\n"
          f"AREA TYPE = {areaType.value}\nSoils (fine/engineered fill): 150–300 mm per loose lift before compaction (default 200 mm).\n"
          f"Coarse rockfill: 300–600 mm per lift (default 400 mm).\n"
          f"Underwater / very soft foundations: thicker rock blanket layers often required (e.g., 500–1,500 mm)\n — treat as a special case based on geotechnical input.\n\n"
          f"TRUCK CAPACITY = {truckCapacity.value} cubic meter\n\n"
          f"ACTUAL DROPS COUNT = {actualDrop.value} drops (Grayed)\n\n"
          f"DUMPING AREA (Length and Width)\nZone A = {dumpWidth('A'):.2f} m\nZone B = {dumpWidth('B'):.2f} m\nZone C = {dumpWidth('C'):.2f} m\n\n"
          f"TOTAL PLANNED DROPS\n"
          f"Zone A = ~{plannedDrops('A')} drops (Brown)\n"
          f"Zone B = ~{plannedDrops('B')} drops (Orange)\n"
          f"Zone C = ~{plannedDrops('C')} drops (Yellow)\n"
          f"Total = ~{plannedDrops_total} drops\n")

  details.value = Detail
  if (aiswitch.value):
    aiDetail = (f"DIMENSION\nHeight/Depth: {mainThickness.value:.2f} m\nWidth: {mainWidth.value:.2f} m\nLength: {mainLength.value:.2f} m\n\n"
          f"ZONE C or TOP FILL = {zoneCType.value} \nConservative Approach: 150–200 mm for grass areas(default 200 mm)\n300–500 mm for planted/tree areas(default 500 mm)\nCustom(default 300 mm))\n\n"
          f"AREA TYPE = {areaType.value}\nSoils (fine/engineered fill): 150–300 mm per loose lift before compaction (default 200 mm).\n"
          f"Coarse rockfill: 300–600 mm per lift (default 400 mm).\n"
          f"Underwater / very soft foundations: thicker rock blanket layers often required (e.g., 500–1,500 mm)\n — treat as a special case based on geotechnical input.\n\n"
          f"TRUCK CAPACITY = {truckCapacity.value} cubic meter\n\n"
          f"DUMPING/DROPPING AREA (Length and Width)\nZone A = {dumpWidth('A'):.2f} m\nZone B = {dumpWidth('B'):.2f} m\nZone C = {dumpWidth('C'):.2f} m\n\n"
          f"TOTAL PLANNED DROPS\n"
          f"Zone A = ~{plannedDrops('A')} drops\n"
          f"Zone B = ~{plannedDrops('B')} drops\n"
          f"Zone C = ~{plannedDrops('C')} drops\n"
          f"Total = ~{plannedDrops_total} drops\n")
    response = gemini_model.generate_content(f"Generate atleast three paragraph insights and recommendation for embankment operation regarding safety, performance, maintenance, and risk management "
          f"using this record {aiDetail}. Include the recommended specific materials to use, the advised implementation steps, and what to avoid. Remove intros and provide directly the insights and recommendation."
          f"result should be in normal format and not in markdown.")
    details.value = Detail + "\n----I N S I G H T S   A N D   R E C O M M E N D A T I O N----\n\n" + response.text

  ## 3. Generate text (chat style)
  #messages = [
  #    {"role": "system", "content": "You are an expert for embankment operation."},
  #    {"role": "user", "content": f"Give a two paragraph recomentdation for this {Detail}"},
  #]
  #inputs = tokenizer.apply_chat_template(
  #    messages,
  #    add_generation_prompt=True,
  #    return_tensors="pt",
  #    return_dict=True,
  #).to(model.device)
  #output = model.generate(**inputs, max_new_tokens=100)
  #recommendation.value = tokenizer.decode(output[0], skip_special_tokens=True)

  #Legend
  ax.text(x_center, y_base, "" # Detail
          , ha="center", va="center", fontsize=8) # Added format specifiers for better readability

  # Styling
  ax.set_xlim(-1, mainWidth.value +3)
  ax.set_ylim(-0.4 - 0.2, y_base + hA + hB + hC + 0.5)
  ax.axis('off')

  #plt.title("Embarkment Information", fontsize=11, weight='bold')
  plt.tight_layout()
  plt.show()

  def on_download(b):
    # 1. Save figure locally
    fig.savefig(file_name, dpi=300, bbox_inches='tight')  # saves as PNG

    # 2. Create a DOCX file and embed the image
    doc = Document()

    # Title (Centered and bold)
    title = doc.add_heading("Embankment Information", level=1)
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER

    # Add image (centered)
    img = doc.add_picture(file_name, width=Inches(5))
    last_paragraph = doc.paragraphs[-1]
    last_paragraph.alignment = WD_ALIGN_PARAGRAPH.CENTER

    # Add details paragraph
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.LEFT  # align text justify

    # Add formatted text (runs)
    run = p.add_run(details.value)
    run.font.size = Pt(12)
    run.font.name = 'Calibri'

    # Add another paragraph with mixed formatting
    p2 = doc.add_paragraph()
    p2.add_run("Generated directly from Google Colab. ").italic = True
    #p2.add_run("This report is automated.").bold = True

    # Optional: add a section break
    #doc.add_page_break()

    # 3. Save the DOCX file
    doc.save(filename.value)

    # 4. Download it
    files.download(filename.value)


  download.on_click(on_download)
  display(details, filename, download)
  hide_spinner()

zoneCType.observe(on_zoneCType_change, names='value')
areaType.observe(on_areaType_change, names='value')
aiswitch.observe(on_aiswitch_change, names='value')

generate.on_click(on_generate)



HTML(value='<Center><h2>Embarkment Calculator</h2></Center>')

HTML(value="<Center><a href='https://chatgpt.com/' target='_blank'>Co-Created by ChatGPT</a></Center>")

HTML(value="<Center><a href='https://drive.google.com/file/d/1hBGz7DkKIq_eJNv2VBrFk-hlh2OVIY0E/view' target='_…

HTML(value="<Center>Try also this App: <a href='https://embankment-calculator.lovable.app/' target='_blank'>Em…

HTML(value='<h2></h2>')

...

FloatSlider(value=0.3, description='Lift Thickness \nbefore Compaction (m):', layout=Layout(width='80%'), max=…

IntText(value=22, description='Truck Capacity (cubic meter):', layout=Layout(width='80%'), style=DescriptionSt…

IntText(value=0, description='Count of Actual Drops (Grayed):', layout=Layout(width='80%'), style=DescriptionS…

ToggleButton(value=False, description='AI(Disabled)', icon='robot', tooltip='Click to enable or disable AI')

Button(description='Generate Image', style=ButtonStyle())